In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]   # COLEPV1
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import sys
import io
import numpy as np
import cv2
from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import ResourceNotFoundError


In [3]:
from colep_ai.core.config import settings

In [4]:

TEST_BLOB_KEY = r"D:\Harpreet Data\1_PROJECTS\Colep_ai\colepV2\outputs\O01_F006_2_Fluxograma_produtivo_Linhas_65_Food_Stamping\pdf_pages_images\O01_F006_2_Fluxograma_produtivo_Linhas_65_Food_Stamping_page_2.png"

In [5]:
CONNECTION_STRING =settings.AZURE_STORAGE_CONNECTION_STRING.get_secret_value()
CONTAINER_NAME = settings.AZURE_STORAGE_CONTAINER_NAME



In [6]:
def make_dummy_png() -> bytes:
    """Creates a small 100x100 green PNG in memory — no disk needed."""
    img = np.zeros((100, 100, 3), dtype=np.uint8)
    img[:] = (0, 200, 0)  # green
    success, buffer = cv2.imencode(".png", img)
    if not success:
        raise RuntimeError("Failed to encode dummy PNG")
    return buffer.tobytes()


In [7]:
# Step 1: Connect
print("Step 1: Connecting to Azure Blob Storage...")
try:
    service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    container_client = service_client.get_container_client(CONTAINER_NAME)
    try:
        container_client.create_container()
        print("  Container created")
    except Exception:
        print("  Container already exists")
    print(f"  Connected | container={CONTAINER_NAME}")

except Exception as e:
    print(f"  FAILED to connect: {e}")
    sys.exit(1)


Step 1: Connecting to Azure Blob Storage...
  Container already exists
  Connected | container=ccsoftblobstorage


In [8]:
  # already hai toh ignore

In [9]:

# Step 2: Make dummy PNG
print("\nStep 2: Creating dummy PNG in memory...")
try:
    png_bytes = make_dummy_png()
    print(f"  Dummy PNG created | size={len(png_bytes)} bytes")
except Exception as e:
    print(f"  FAILED to create dummy PNG: {e}")
    sys.exit(1)


Step 2: Creating dummy PNG in memory...
  Dummy PNG created | size=330 bytes


In [10]:

# Step 3: Upload
print(f"\nStep 3: Uploading to blob key → {TEST_BLOB_KEY}")
try:
    container_client.upload_blob(
        name=TEST_BLOB_KEY,
        data=io.BytesIO(png_bytes),
        overwrite=True,
        content_settings=None,
    )
    print(f"  Upload SUCCESS")
except Exception as e:
    print(f"  FAILED to upload: {e}")
    sys.exit(1)


Step 3: Uploading to blob key → D:\Harpreet Data\1_PROJECTS\Colep_ai\colepV2\outputs\O01_F006_2_Fluxograma_produtivo_Linhas_65_Food_Stamping\pdf_pages_images\O01_F006_2_Fluxograma_produtivo_Linhas_65_Food_Stamping_page_2.png
  Upload SUCCESS


In [11]:
# Step 4: Verify exists
print("\nStep 4: Verifying blob exists...")
try:
    blob_client = container_client.get_blob_client(TEST_BLOB_KEY)
    props = blob_client.get_blob_properties()
    print(f"  Exists | size={props.size} bytes | key={TEST_BLOB_KEY}")
except ResourceNotFoundError:
    print(f"  FAILED — blob not found after upload. Something is wrong.")
    sys.exit(1)
except Exception as e:
    print(f"  FAILED to verify: {e}")
    sys.exit(1)



Step 4: Verifying blob exists...


  Exists | size=330 bytes | key=D:\Harpreet Data\1_PROJECTS\Colep_ai\colepV2\outputs\O01_F006_2_Fluxograma_produtivo_Linhas_65_Food_Stamping\pdf_pages_images\O01_F006_2_Fluxograma_produtivo_Linhas_65_Food_Stamping_page_2.png


In [12]:

# Step 5: Delete
print("\nStep 5: Deleting test blob...")
try:
    blob_client.delete_blob()
    print(f"  Deleted")
except Exception as e:
    print(f"  FAILED to delete: {e}")
    sys.exit(1)




Step 5: Deleting test blob...
  Deleted


In [13]:

# Step 6: Verify gone
print("\nStep 6: Verifying blob is deleted...")
try:
    blob_client.get_blob_properties()
    print(f"  FAILED — blob still exists after delete. Something is wrong.")
    sys.exit(1)
except ResourceNotFoundError:
    print(f"  Confirmed deleted")
except Exception as e:
    print(f"  FAILED to verify deletion: {e}")
    sys.exit(1)


Step 6: Verifying blob is deleted...
  Confirmed deleted


In [14]:
print("\n=== ALL STEPS PASSED — Blob connection is working ===\n")


=== ALL STEPS PASSED — Blob connection is working ===



In [15]:



def run_test():
    print("\n=== Azure Blob Storage Connection Test ===\n")

    # Step 1: Connect
    print("Step 1: Connecting to Azure Blob Storage...")
    try:
        service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
        container_client = service_client.get_container_client(CONTAINER_NAME)
        print(f"  Connected | container={CONTAINER_NAME}")
    except Exception as e:
        print(f"  FAILED to connect: {e}")
        sys.exit(1)

    # Step 2: Make dummy PNG
    print("\nStep 2: Creating dummy PNG in memory...")
    try:
        png_bytes = make_dummy_png()
        print(f"  Dummy PNG created | size={len(png_bytes)} bytes")
    except Exception as e:
        print(f"  FAILED to create dummy PNG: {e}")
        sys.exit(1)

    # Step 3: Upload
    print(f"\nStep 3: Uploading to blob key → {TEST_BLOB_KEY}")
    try:
        container_client.upload_blob(
            name=TEST_BLOB_KEY,
            data=io.BytesIO(png_bytes),
            overwrite=True,
            content_settings=None,
        )
        print(f"  Upload SUCCESS")
    except Exception as e:
        print(f"  FAILED to upload: {e}")
        sys.exit(1)

    # Step 4: Verify exists
    print("\nStep 4: Verifying blob exists...")
    try:
        blob_client = container_client.get_blob_client(TEST_BLOB_KEY)
        props = blob_client.get_blob_properties()
        print(f"  Exists | size={props.size} bytes | key={TEST_BLOB_KEY}")
    except ResourceNotFoundError:
        print(f"  FAILED — blob not found after upload. Something is wrong.")
        sys.exit(1)
    except Exception as e:
        print(f"  FAILED to verify: {e}")
        sys.exit(1)

    # Step 5: Delete
    print("\nStep 5: Deleting test blob...")
    try:
        blob_client.delete_blob()
        print(f"  Deleted")
    except Exception as e:
        print(f"  FAILED to delete: {e}")
        sys.exit(1)

    # Step 6: Verify gone
    print("\nStep 6: Verifying blob is deleted...")
    try:
        blob_client.get_blob_properties()
        print(f"  FAILED — blob still exists after delete. Something is wrong.")
        sys.exit(1)
    except ResourceNotFoundError:
        print(f"  Confirmed deleted")
    except Exception as e:
        print(f"  FAILED to verify deletion: {e}")
        sys.exit(1)

    print("\n=== ALL STEPS PASSED — Blob connection is working ===\n")



In [16]:
# run_test()